# Modelo Generativo de Texto

En esta sección se aborda la parte generativa del proyecto, cuyo objetivo es desarrollar un modelo capaz de aprender el estilo de escritura de los distintos miembros del grupo a partir de los mensajes de WhatsApp.

La idea inicial consistía en construir un único modelo generativo condicionado, en el que se incluyera la identidad del usuario como parte de la secuencia de entrada. De este modo, el modelo sería capaz de generar texto adaptado al estilo de cualquier miembro del grupo indicando el usuario deseado.

In [1]:
import numpy as np
import pandas as pd
import re
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GRU
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/TRABAJO_NO_ESTRUCTURADOS/TRABAJO_NO_ESTRUCTURADOS/data/dataframe_concatenado.csv')

df.head()

,timestamp,person,message_combined
0,2022-07-10 20:14:08,Paula,me estoy agobiando con los grupos asiq hablamo...
1,2022-07-10 20:14:12,Carmen,coño gracias jajjjajajj a ver si solo hay un v...
2,2022-07-10 20:14:59,Paula,yo voy al baño y voy para all
3,2022-07-10 20:15:00,Carmen,aunq la gente suele pedir por glovo
4,2022-07-10 20:15:05,Paula,asiq me da igual la hora


In [4]:
# nombres consistentes
df['person'] = df['person'].astype(str).str.strip().str.lower()

# texto
df['message'] = df['message_combined'].astype(str).str.lower()

# eliminar urls
df['message'] = df['message'].apply(lambda x: re.sub(r'http\S+', '', x))

# quitar espacios extra
df['message'] = df['message'].str.strip()

In [5]:
counts = df['person'].value_counts()

# mínimo 50 mensajes (ajusta si quieres)
valid_users = counts[counts > 50].index

df = df[df['person'].isin(valid_users)]

print(df['person'].value_counts())

person
claudia    8870
paula      8437
carmen     7219
angela     5788
Name: count, dtype: int64


## Enfoque inicial: modelo condicionado por usuario

El planteamiento original consistía en entrenar un modelo de generación de texto que incorporase el usuario como parte del input, por ejemplo:

[usuario] mensaje

De esta forma, el modelo aprendería no solo la estructura del lenguaje, sino también patrones específicos de cada usuario, como expresiones habituales, tono, longitud de frases o uso de emojis.

Esto permitiría generar texto condicionado, por ejemplo:

Input: "[claudia] tio"  
Output: "tio que haces hoy jajaja"

In [6]:
texts = []

for _, row in df.iterrows():
    user = row['person']
    msg = row['message']

    # ignorar mensajes muy cortos
    if len(msg.split()) > 2:
        texts.append(f"[{user}] {msg}")

print("Total frases:", len(texts))

Total frases: 24545


In [7]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

total_words = max(5000, len(tokenizer.word_index) + 1)

print("Vocab size:", total_words)

Vocab size: 23350


In [8]:
input_sequences = []

for line in texts:
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

In [9]:
max_sequence_len = max(len(x) for x in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

In [10]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = to_categorical(y, num_classes=total_words)

In [11]:
model = Sequential()

model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
model.add(LSTM(150, return_sequences=True))
model.add(LSTM(100))
model.add(Dropout(0.2))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Problema encontrado: limitaciones computacionales

Sin embargo, este enfoque presenta un elevado coste computacional. Al entrenar un único modelo con todos los usuarios, el volumen de datos y el tamaño del vocabulario aumentan considerablemente, lo que incrementa el tiempo de entrenamiento y los requisitos de memoria.

Durante la ejecución en Google Colab, se produjeron interrupciones debido a limitaciones de recursos, finalizando la sesión antes de completar el entrenamiento.

Además, al intentar ejecutar el modelo en local, el tiempo estimado de entrenamiento era de aproximadamente 4 horas por época, lo que hacía inviable completar el proceso en un tiempo razonable.

In [12]:
# history = model.fit(X, y, epochs=10, batch_size=128)

## Solución adoptada

Debido a estas limitaciones, se optó por una solución alternativa: entrenar modelos generativos independientes para cada usuario.

Cada modelo se entrena únicamente con los mensajes de un miembro del grupo, lo que reduce significativamente el tamaño del dataset, el vocabulario y el tiempo de entrenamiento.

## LSTM

In [13]:
def train_model_for_user(user, epochs=10, base_path="/content/drive/MyDrive/TRABAJO_NO_ESTRUCTURADOS/TRABAJO_NO_ESTRUCTURADOS/models/"):

    import os
    import pickle

    print(f"\nEntrenando modelo para: {user}")

    # crear carpeta si no existe
    os.makedirs(base_path, exist_ok=True)

    # filtrar usuario
    df_user = df[df['person'] == user]

    texts = [t for t in df_user['message'] if len(t.split()) >= 4]

    print(f"Número de textos: {len(texts)}")

    # tokenizer (IMPORTANTE: sin OOV para generación)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(texts)

    total_words = len(tokenizer.word_index) + 1

    # secuencias
    input_sequences = []

    for line in texts:
        token_list = tokenizer.texts_to_sequences([line])[0]

        for i in range(1, len(token_list)):
            input_sequences.append(token_list[:i+1])

    # padding
    max_sequence_len = max(len(x) for x in input_sequences)

    input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

    # X / y
    X = input_sequences[:, :-1]
    y = input_sequences[:, -1]

    y = to_categorical(y, num_classes=total_words)

    # modelo (ligero pero bueno)
    model = Sequential()
    model.add(Embedding(total_words, 100))
    model.add(LSTM(100))
    model.add(Dropout(0.2))
    model.add(Dense(total_words, activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    # early stopping
    early_stop = EarlyStopping(
        monitor='loss',
        patience=1,
        min_delta=0.001,
        restore_best_weights=True
    )
    # entrenar
    model.fit(X, y, epochs=epochs, batch_size=128, callbacks=[early_stop])

    # GUARDADO AUTOMÁTICO
    print(f"Guardando modelo de {user}...")

    # guardar modelo
    model.save(base_path + f"{user}_lstm_model.h5")

    # guardar tokenizer
    with open(base_path + f"{user}_lstm_tokenizer.pkl", "wb") as f:
        pickle.dump(tokenizer, f)

    # guardar max_len
    with open(base_path + f"{user}_lstm_maxlen.pkl", "wb") as f:
        pickle.dump(max_sequence_len, f)

    print(f"Modelo de {user} guardado correctamente")

    return model, tokenizer, max_sequence_len

In [14]:
users = [] #["claudia", "paula", "angela", "carmen"]

models = {}
tokenizers = {}
max_lens = {}

for user in users:
    model, tokenizer, max_len = train_model_for_user(user, epochs=50)

    models[user] = model
    tokenizers[user] = tokenizer
    max_lens[user] = max_len

## Pruebas de generación

Para hacer las pruebas de generación, primero cargamos los modelos almacenados post entrenamiento

In [15]:
import os
import pickle
from tensorflow.keras.models import load_model

base_path = "/content/drive/MyDrive/TRABAJO_NO_ESTRUCTURADOS/TRABAJO_NO_ESTRUCTURADOS/models/"

users = ["claudia", "paula", "carmen", "angela"]

for user in users:

    print(f"Cargando datos de: {user}")

    # rutas
    model_path = os.path.join(base_path, f"{user}_lstm_model.h5")
    tokenizer_path = os.path.join(base_path, f"{user}_lstm_tokenizer.pkl")
    maxlen_path = os.path.join(base_path, f"{user}_lstm_maxlen.pkl")

    # cargar modelo
    models[user] = load_model(model_path)

    # cargar tokenizer
    with open(tokenizer_path, "rb") as f:
        tokenizers[user] = pickle.load(f)

    # cargar max_len
    with open(maxlen_path, "rb") as f:
        max_lens[user] = pickle.load(f)

print("Todos los modelos cargados correctamente")

Cargando datos de: claudia


Cargando datos de: paula


Cargando datos de: carmen


Cargando datos de: angela


Todos los modelos cargados correctamente


In [16]:
def generate_text(model, tokenizer, max_len, seed_text, next_words=10):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

        predicted = model.predict(token_list, verbose=0)[0] # Get the first (and only) row of probabilities
        predicted_word = np.random.choice(len(tokenizer.word_index) + 1, p=predicted) # Use total_words from tokenizer

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_word:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

In [17]:
def generate_by_user(user, seed_text):

    model = models[user]
    tokenizer = tokenizers[user]
    max_len = max_lens[user]

    return generate_text(model, tokenizer, max_len, seed_text)

Probamos varias frases

In [29]:
print("Claudia:", generate_by_user("claudia", "tia que no puedo que estoy"))
print("Carmen:", generate_by_user("carmen", "tia que no puedo que estoy"))
print("Paula:", generate_by_user("paula", "tia que no puedo que estoy"))
print("Angela:", generate_by_user("angela", "tia que no puedo que estoy"))

Claudia: tia que no puedo que estoy entendiendo un examen ahora esa la uni inició una videollamada
Carmen: tia que no puedo que estoy en la mochila y no os salgatan caro desde como
Paula: tia que no puedo que estoy atascadisima con todas formas con el fiesta en el culo
Angela: tia que no puedo que estoy gastando mazo pillar la comida q quieres algún tiempo jajajajja


In [30]:
print("Claudia:", generate_by_user("claudia", "tia no puedo ir hoy porque"))
print("Carmen:", generate_by_user("carmen", "tia no puedo ir hoy porque"))
print("Paula:", generate_by_user("paula", "tia no puedo ir hoy porque"))
print("Angela:", generate_by_user("angela", "tia no puedo ir hoy porque"))

Claudia: tia no puedo ir hoy porque creo y de lo q ha hablado mal pero oye
Carmen: tia no puedo ir hoy porque yo entonces no puedo reservar no te queda genial y
Paula: tia no puedo ir hoy porque es lo mismo carmen enplan se muero no puedo dormirrr
Angela: tia no puedo ir hoy porque luego os echo de haberos chino el día reclamando pues


In [31]:
print("Claudia:", generate_by_user("claudia", "tias no sabeis"))
print("Carmen:", generate_by_user("carmen", "tias no sabeis"))
print("Paula:", generate_by_user("paula", "tias no sabeis"))
print("Angela:", generate_by_user("angela", "tias no sabeis"))

Claudia: tias no sabeis q estoy acompañándole de una cuerda q mas como q
Carmen: tias no sabeis q marc dicen q queraisss creo q no les digo
Paula: tias no sabeis bueno el ej jakjajajaj en el piso que ademas al
Angela: tias no sabeis lo q dices y ahora no osea q no puedo


In [32]:
print("Claudia:", generate_by_user("claudia", "hacemos algo"))
print("Carmen:", generate_by_user("carmen", "hacemos algo"))
print("Paula:", generate_by_user("paula", "hacemos algo"))
print("Angela:", generate_by_user("angela", "hacemos algo"))

Claudia: hacemos algo duro e eres mala ajajajaj si nos hace llevamos seguro
Carmen: hacemos algo quieren hacer el mismo proyecto de hacerlo al llegar a
Paula: hacemos algo baratillooo como en persona mañana me ha fijado por susurrar
Angela: hacemos algo madre mia no sabía q mi cartilla d darle pena


Aunque el modelo LSTM ha generado resultados razonables, se observan ciertas limitaciones en la calidad y coherencia del texto generado.

Esto puede deberse, entre otros factores, al tamaño del dataset y a la complejidad del modelo, que puede dificultar el aprendizaje efectivo en contextos con menor cantidad de datos.

Por ello, se decide explorar una arquitectura alternativa: las **Gated Recurrent Units (GRU)**. Este tipo de modelo, similar a LSTM pero con una estructura más simple y un menor número de parámetros, suele adaptarse mejor a conjuntos de datos más pequeños, permitiendo un entrenamiento más eficiente y una mejor generalización en tareas de generación de texto.

A continuación, se entrena un modelo GRU para comparar cualitativamente los resultados generados.

## GRU

In [22]:
def train_gru_for_user(user, epochs=10, base_path="/content/drive/MyDrive/TRABAJO_NO_ESTRUCTURADOS/TRABAJO_NO_ESTRUCTURADOS/models/"):

    import os
    import pickle

    print(f"\nEntrenando modelo para: {user}")

    # crear carpeta si no existe
    os.makedirs(base_path, exist_ok=True)

    # filtrar usuario
    df_user = df[df['person'] == user]

    texts = [t for t in df_user['message'] if len(t.split()) >= 4]

    print(f"Número de textos: {len(texts)}")

    # tokenizer (IMPORTANTE: sin OOV para generación)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(texts)

    total_words = len(tokenizer.word_index) + 1

    # secuencias
    input_sequences = []

    for line in texts:
        token_list = tokenizer.texts_to_sequences([line])[0]

        for i in range(1, len(token_list)):
            input_sequences.append(token_list[:i+1])

    # padding
    max_sequence_len = max(len(x) for x in input_sequences)

    input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

    # X / y
    X = input_sequences[:, :-1]
    y = input_sequences[:, -1]

    y = to_categorical(y, num_classes=total_words)

    # modelo (ligero pero bueno)
    model = Sequential()
    model.add(Embedding(total_words, 100))
    model.add(GRU(100))
    model.add(Dropout(0.2))
    model.add(Dense(total_words, activation='softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    # early stopping
    early_stop = EarlyStopping(
        monitor='loss',
        patience=1,
        min_delta=0.001,
        restore_best_weights=True
    )
    # entrenar
    model.fit(X, y, epochs=epochs, batch_size=128, callbacks=[early_stop])

    # GUARDADO AUTOMÁTICO
    print(f"Guardando modelo de {user}...")

    # guardar modelo
    model.save(base_path + f"{user}_gru_model.h5")

    # guardar tokenizer
    with open(base_path + f"{user}_gru_tokenizer.pkl", "wb") as f:
        pickle.dump(tokenizer, f)

    # guardar max_len
    with open(base_path + f"{user}_gru_maxlen.pkl", "wb") as f:
        pickle.dump(max_sequence_len, f)

    print(f"Modelo de {user} guardado correctamente")

    return model, tokenizer, max_sequence_len

In [23]:
users = [] #["claudia", "paula", "angela", "carmen"]

for user in users:
    model, tokenizer, max_len = train_gru_for_user(user, epochs=50)
    usergru = user + "_gru"
    models[usergru] = model
    tokenizers[usergru] = tokenizer
    max_lens[usergru] = max_len

Por cuestiones de tiempo de entrenamiento, el modelo GRU solo ha podido ser entrenado para el usuario de "claudia". Por tanto, las siguientes comparaciones de los dos modelos serán para ese usuario

# Pruebas de generación

Cargamos el modelo GRU entrenado previamente para las comparaciones

In [24]:
import os
import pickle
from tensorflow.keras.models import load_model

base_path = "/content/drive/MyDrive/TRABAJO_NO_ESTRUCTURADOS/TRABAJO_NO_ESTRUCTURADOS/models/"

users = ["claudia"]

for user in users:

    print(f"Cargando datos de: {user}")

    # rutas
    model_path = os.path.join(base_path, f"{user}_gru_model.h5")
    tokenizer_path = os.path.join(base_path, f"{user}_gru_tokenizer.pkl")
    maxlen_path = os.path.join(base_path, f"{user}_gru_maxlen.pkl")

    usergru = user + "_gru"
    # cargar modelo
    models[usergru] = load_model(model_path)

    # cargar tokenizer
    with open(tokenizer_path, "rb") as f:
        tokenizers[usergru] = pickle.load(f)

    # cargar max_len
    with open(maxlen_path, "rb") as f:
        max_lens[usergru] = pickle.load(f)

print("Todos los modelos cargados correctamente")

Cargando datos de: claudia


Todos los modelos cargados correctamente


In [25]:
print("Claudia (LSTM):", generate_by_user("claudia", "tia que no puedo que estoy"))
print("Claudia (GRU):", generate_by_user("claudia_gru", "tia que no puedo que estoy"))

Claudia (LSTM): tia que no puedo que estoy de rescaa a uno más cuantas del al zara en
Claudia (GRU): tia que no puedo que estoy sentada días en la bolsita transparente no m hacen na


In [26]:
print("Claudia (LSTM):", generate_by_user("claudia", "tia no puedo ir hoy porque"))
print("Claudia (GRU):", generate_by_user("claudia_gru", "tia no puedo ir hoy porque"))

Claudia (LSTM): tia no puedo ir hoy porque a volver con pau y me vaya lo has puesto
Claudia (GRU): tia no puedo ir hoy porque me he agoviado por el tema y m suspende igual


In [27]:
print("Claudia (LSTM):", generate_by_user("claudia", "tias no sabeis"))
print("Claudia (GRU):", generate_by_user("claudia_gru", "tias no sabeis"))

Claudia (LSTM): tias no sabeis q le he dado cuenta mal q lau suspendio y
Claudia (GRU): tias no sabeis cuando digas por lo q te jode con las del


In [28]:
print("Claudia (LSTM):", generate_by_user("claudia", "hacemos algo"))
print("Claudia (GRU):", generate_by_user("claudia_gru", "hacemos algo"))

Claudia (LSTM): hacemos algo pero q nos vemos el flash y no entiendo pillo
Claudia (GRU): hacemos algo no pienso bajo yo voy con ahi quieres elefantes pero


## Análisis de resultados

A partir de los ejemplos generados, se puede observar que el modelo basado en GRU produce frases que, en general, presentan una mayor coherencia y fluidez inicial en comparación con las generadas por el modelo LSTM.

Sin embargo, a medida que se incrementa la longitud del texto generado, ambos modelos tienden a perder consistencia, generando secuencias menos coherentes y con menor sentido global. Este comportamiento es habitual en modelos secuenciales tradicionales, que tienen dificultades para mantener dependencias a largo plazo en el texto.

Para abordar estas limitaciones, en el siguiente notebook se implementa un modelo basado en **Transformers preentrenados**, que incorporan el mecanismo de atención. Este enfoque permite capturar mejor las relaciones entre palabras dentro de la secuencia, por lo que se espera obtener resultados más coherentes y de mayor calidad en la generación de texto.